# Trust Layer — Uzbek Car Recognizer

Turns the 92% ConvNeXt into a system that **knows when it doesn't know**, so it can hold a
very high precision by *abstaining* on the hard cases.

Steps: **temperature-scale** the confidences → find the threshold for **≥99% precision** on
validation → apply it **once** to the sealed test → reliability diagram + a grid of the hardest
mistakes.

**Prerequisite:** run the Model Gate through Cells 11–12 first, so `dataset_split.zip` **and**
`modelgate_artifacts.zip` are both in Drive `CapstoneCars/`. Set **Runtime → T4 GPU**.

In [ ]:
# ── Cell 1 · Setup ──────────────────────────────────────────────
!pip install -q timm
import torch, torch.nn.functional as F, numpy as np, json, glob
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import timm
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", dev)

In [ ]:
# ── Cell 2 · Reload the trained model + the split from Drive ────
from google.colab import drive
drive.mount('/content/drive')
ds = glob.glob('/content/drive/MyDrive/**/dataset_split.zip', recursive=True)[0]
ar = glob.glob('/content/drive/MyDrive/**/modelgate_artifacts.zip', recursive=True)[0]
print("data:", ds); print("artifacts:", ar)
!rm -rf /content/dataset /content/artifacts
!unzip -o -q "$ds" -d /content
!unzip -o -q "$ar" -d /content

cfg = json.load(open('/content/artifacts/config.json'))
CLASSES = cfg['classes']
print("winner:", cfg['winner'], "| classes:", CLASSES)
model = timm.create_model(cfg['model_id'], pretrained=False, num_classes=len(CLASSES))
model.load_state_dict(torch.load('/content/artifacts/model.pt', map_location=dev))
model.to(dev).eval()
print("model reloaded ✓")

In [ ]:
# ── Cell 3 · Val/test loaders + raw logits ──────────────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
IMG = cfg['img_size']; MEAN, STD = tuple(cfg['mean']), tuple(cfg['std'])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_ds  = datasets.ImageFolder('/content/dataset/val',  eval_tf)
test_ds = datasets.ImageFolder('/content/dataset/test', eval_tf)
assert val_ds.classes == CLASSES, "class order mismatch!"
val_loader  = DataLoader(val_ds,  64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, 64, shuffle=False, num_workers=2)

@torch.no_grad()
def get_logits(loader):
    L, Y = [], []
    for x,y in loader:
        L.append(model(x.to(dev)).cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)
val_logits, val_y = get_logits(val_loader)
test_logits, test_y = get_logits(test_loader)
print("val", tuple(val_logits.shape), "| test", tuple(test_logits.shape))

In [ ]:
# ── Cell 4 · Temperature scaling (calibrate the confidences) ────
# Fit ONE scalar T on validation to make the softmax probabilities honest (Guo et al. 2017).
T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.05, max_iter=200)
def closure():
    opt.zero_grad(); loss = F.cross_entropy(val_logits/T, val_y); loss.backward(); return loss
opt.step(closure)
T = max(0.05, float(T.detach()))
print("fitted temperature T =", round(T, 3))

def ece(logits, y, T=1.0, bins=15):
    p = torch.softmax(logits/T, 1); conf, pred = p.max(1)
    conf, pred, y = conf.numpy(), pred.numpy(), y.numpy()
    e, edges = 0.0, np.linspace(0, 1, bins+1)
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i+1])
        if m.sum(): e += abs((pred[m]==y[m]).mean() - conf[m].mean()) * m.mean()
    return e
print(f"ECE (test) — before {ece(test_logits,test_y,1.0):.3f}   after {ece(test_logits,test_y,T):.3f}")

In [ ]:
# ── Cell 5 · Find the ≥99%-precision threshold on VALIDATION ────
# Selective prediction: only answer when calibrated confidence >= threshold; else abstain.
TARGET = 0.99
def conf_pred(logits, T):
    p = torch.softmax(logits/T, 1); c, pr = p.max(1); return c.numpy(), pr.numpy()

vc, vp = conf_pred(val_logits, T); v_ok = (vp == val_y.numpy())
best_t, best_cov = None, -1.0
for t in np.unique(vc):
    m = vc >= t
    if m.sum() >= 20 and v_ok[m].mean() >= TARGET and m.mean() > best_cov:
        best_cov, best_t = m.mean(), float(t)

if best_t is None:
    # 99% not reachable — report the best precision we can and its coverage
    accs = [( (vc>=t).mean(), v_ok[vc>=t].mean(), t) for t in np.unique(vc) if (vc>=t).sum()>=20]
    cov, pr, best_t = max(accs, key=lambda z: z[1])
    print(f"⚠️ {TARGET:.0%} precision not reachable on val. Best: {pr:.1%} precision @ {cov:.1%} coverage (T={best_t:.3f}).")
else:
    print(f"threshold for ≥{TARGET:.0%} precision (chosen on VAL): {best_t:.3f}")
    print(f"  → val coverage {(vc>=best_t).mean():.1%}, val precision {v_ok[vc>=best_t].mean():.1%}")

In [ ]:
# ── Cell 6 · Apply the val-chosen threshold ONCE to the sealed test ──
tc, tp = conf_pred(test_logits, T); t_ok = (tp == test_y.numpy())
m = tc >= best_t
print(f"SEALED TEST @ threshold {best_t:.3f}:")
print(f"  precision on answered : {t_ok[m].mean():.1%}   (of the cars it answers, this fraction are correct)")
print(f"  coverage (answered)   : {m.mean():.1%}")
print(f"  abstained → human     : {(~m).mean():.1%}")
print(f"  (for reference: no-abstain accuracy = {t_ok.mean():.1%} at 100% coverage)")

# precision–coverage (risk–coverage) curve on test
cov, prec = [], []
for t in np.linspace(tc.min(), tc.max(), 120):
    mm = tc >= t
    if mm.sum() >= 10: cov.append(mm.mean()); prec.append(t_ok[mm].mean())
plt.figure(figsize=(6,4))
plt.plot(cov, prec, '-')
plt.axhline(TARGET, color='r', ls='--', label=f'{TARGET:.0%} precision')
plt.scatter([m.mean()], [t_ok[m].mean()], color='k', zorder=5, label='operating point')
plt.xlabel('coverage (fraction answered)'); plt.ylabel('precision on answered')
plt.title('Precision–coverage — sealed test'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 7 · Reliability diagrams (before vs after calibration) ─
def reliability(logits, y, T, ax, title):
    p = torch.softmax(logits/T, 1); conf, pred = p.max(1)
    conf, pred, y = conf.numpy(), pred.numpy(), y.numpy()
    edges = np.linspace(0, 1, 11); xs, ys = [], []
    for i in range(10):
        mm = (conf > edges[i]) & (conf <= edges[i+1])
        if mm.sum(): xs.append(conf[mm].mean()); ys.append((pred[mm]==y[mm]).mean())
    ax.plot([0,1],[0,1],'k--',alpha=.5); ax.plot(xs, ys, 'o-')
    ax.set_title(title); ax.set_xlabel('confidence'); ax.set_ylabel('accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
fig, axs = plt.subplots(1, 2, figsize=(9,4))
reliability(test_logits, test_y, 1.0, axs[0], 'before (T=1)')
reliability(test_logits, test_y, T,   axs[1], f'after (T={T:.2f})')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 8 · Error analysis: the most confident MISTAKES ────────
wrong = np.where(tp != test_y.numpy())[0]
wrong = wrong[np.argsort(-tc[wrong])][:12]      # highest-confidence errors = the instructive ones
fig, ax = plt.subplots(2, 6, figsize=(15, 5.5))
for a, idx in zip(ax.ravel(), wrong):
    path, _ = test_ds.samples[idx]
    a.imshow(Image.open(path)); a.axis('off')
    a.set_title(f"true {CLASSES[int(test_y[idx])]}\npred {CLASSES[tp[idx]]} ({tc[idx]:.2f})", fontsize=8)
for a in ax.ravel()[len(wrong):]: a.axis('off')
plt.suptitle("Most confident test mistakes — mostly the Cobalt/Gentra/Nexia look-alikes")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9 · Save calibration to the artifact + back up ─────────
cfg['temperature'] = float(T)
cfg['abstain_threshold'] = float(best_t)
json.dump(cfg, open('/content/artifacts/config.json','w'), indent=2)
!cd /content && zip -q -r modelgate_artifacts.zip artifacts >/dev/null
!cp /content/modelgate_artifacts.zip /content/drive/MyDrive/CapstoneCars/
print("✅ temperature + abstain threshold saved to config.json and backed up to Drive")
print("   config now:", {k: cfg[k] for k in ['winner','temperature','abstain_threshold']})